In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from skimage import io
from skimage import data
from skimage.transform import resize
from skimage.color import rgb2gray
from scipy.ndimage import uniform_filter

In [ ]:
# Load images

left, right, _ = data.stereo_motorcycle()
left = rgb2gray(left)
right = rgb2gray(right)

# noise = np.random.normal(0, 0.5, left.shape)
# left = np.clip(left + noise, 0, 1)
# right = np.clip(right + noise, 0, 1)


scale = 0.3  # 30% size
left = resize(left, (int(left.shape[0]*scale), int(left.shape[1]*scale)),
                    anti_aliasing=True)
right = resize(right, (int(right.shape[0]*scale), int(right.shape[1]*scale)),
                     anti_aliasing=True)



plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.title("Left")
plt.imshow(left, cmap='gray')

plt.subplot(1,2,2)
plt.title("Right")
plt.imshow(right, cmap='gray')
plt.show()

In [ ]:
# Sum of Squared Differences 
def compute_cost_volume(left, right, max_disp=20, window=5):
    h, w = left.shape
    cost_volume = np.zeros((h, w, max_disp))

    for d in range(max_disp):
        shifted = np.roll(right, d, axis=1)
        
        diff = (left - shifted) ** 2
        cost = uniform_filter(diff, size=window)
        
        cost_volume[:, :, d] = cost
        
    return cost_volume

In [ ]:
# Block Matching
def block_matching(cost_volume):
    return np.argmin(cost_volume, axis=2)

cost_volume = compute_cost_volume(left, right)
disp_block = block_matching(cost_volume)

plt.imshow(disp_block, cmap='plasma')
plt.title("Block Matching Disparity")
plt.colorbar()
plt.show()

In [ ]:
# Dynamic Programming (Scanline Stereo)
def dp_scanline(cost_volume, smoothness=0.001):
    h, w, D = cost_volume.shape
    disp = np.zeros((h, w), dtype=np.int32)

    V = lambda d1, d2: smoothness * abs(d1 - d2)

    for y in range(h):
        dp = np.zeros((w, D))

        # init
        dp[0] = cost_volume[y, 0]

        for x in range(1, w):
            for d in range(D):

                best = 1e9

                for d2 in range(D):
                    val = dp[x-1, d2] + cost_volume[y, x, d] + V(d2, d)
                    best = min(best, val)

                dp[x, d] = best

        disp[y] = np.argmin(dp, axis=1)

    return disp

disp_dp = dp_scanline(cost_volume)

plt.imshow(disp_dp, cmap='plasma')
plt.title("Dynamic Programming Disparity")
plt.colorbar()
plt.show()

In [ ]:
# Dense Stereo MRF
def mrf_stereo(cost_volume, lam=2.0, iterations=5):
    h, w, D = cost_volume.shape

    disp = np.argmin(cost_volume, axis=2)

    for _ in range(iterations):
        new_disp = disp.copy()

        for y in range(1, h-1):
            for x in range(1, w-1):

                best_d = disp[y, x]
                best_E = 1e9

                for d in range(D):

                    unary = cost_volume[y, x, d]

                    binary = 0
                    for dy, dx in [(-1,0),(1,0),(0,-1),(0,1)]:
                        binary += abs(d - disp[y+dy, x+dx])

                    E = unary + lam * binary

                    if E < best_E:
                        best_E = E
                        best_d = d

                new_disp[y, x] = best_d

        disp = new_disp

    return disp

disp_mrf = mrf_stereo(cost_volume)

plt.imshow(disp_mrf, cmap='plasma')
plt.title("MRF Disparity")
plt.colorbar()
plt.show()

In [ ]:
# Comparison
plt.figure(figsize=(15,3))

plt.subplot(1,3,1)
plt.title("Block")
plt.imshow(disp_block, cmap='plasma')

plt.subplot(1,3,2)
plt.title("DP")
plt.imshow(disp_dp, cmap='plasma')

plt.subplot(1,3,3)
plt.title("MRF")
plt.imshow(disp_mrf, cmap='plasma')

plt.show()